In [ ]:
# Install the PyPharao code and py3Dmol
!pip install rdkit
!pip install git+https://github.com/silicos-it/PyPharao.git
!pip install mols2grid py3Dmol

In [ ]:
# Import all required modules
from tqdm import tqdm_notebook as tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import rdFMCS, rdMolAlign, rdMolTransforms

from IPython.display import display
from IPython.display import SVG

import py3Dmol
from ipywidgets import widgets

import mols2grid
from pypharao import *

from pathlib import Path
import requests

In [ ]:
# Build the pharmacophore query from phenol
phenol = Chem.AddHs(Chem.MolFromSmiles("c1ccccc1O"))
AllChem.EmbedMolecule(phenol)
AllChem.UFFOptimizeMolecule(phenol)

pharmacophore = query_pharmacophore_from_molecule(phenol, name="phenol")
print(f"\nQuery {pharmacophore.get_name()!r} ({len(pharmacophore)} features):")
for p in pharmacophore:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
# Create 3D-conformations of both phenol and 7 other structures

smiles = ["c1ccccc1O",
    "Cc1ccc(CO)s1",
    "Cc1nc(CO)n[nH]1",
    "O=C(O)c1c(F)cc(F)cc1F",
    "Nc1cc(F)ccc1C(=O)O",
    "Cc1ccc(Cl)cc1CN",
    "NCc1cccc(N)n1",
    "Cc1conc1CC(=O)O"]
mols = [Chem.AddHs(Chem.MolFromSmiles(s)) for s in smiles]
for mol in mols:
    AllChem.EmbedMolecule(mol)
    AllChem.MMFFOptimizeMolecule(mol)

In [ ]:
# Determine MCSS between all mols

NUM_CONFS = 5

# MCSS
res = rdFMCS.FindMCS(mols)
mcs_pattern = Chem.MolFromSmarts(res.smartsString)

# Generate conformations of all molecules starting from molecule 1 (not phenol)
for mol in mols[1:]:
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=NUM_CONFS)
    for cid in cids:
        AllChem.MMFFOptimizeMolecule(mol, confId=cid)

# Align all molecules to the first molecule (reference)
ref_mol = mols[0]
ref_idx = ref_mol.GetSubstructMatch(mcs_pattern)

for mol in mols[1:]:

    # Find matching atoms for the current molecule
    mol_idx = mol.GetSubstructMatch(mcs_pattern)
    atom_map = list(zip(mol_idx, ref_idx))
    ref_cid = 0  # reference has a single conformer

    per_conf_rmsd: list[float] = []
    for cid in range(mol.GetNumConformers()):
        rmsd, trans_matrix = rdMolAlign.GetAlignmentTransform(
            mol,
            ref_mol,
            prbCid=cid,
            refCid=ref_cid,
            atomMap=atom_map,
        )
        rdMolTransforms.TransformConformer(mol.GetConformer(cid), trans_matrix)
        per_conf_rmsd.append(rmsd)

    print(
        "Aligned molecule. Per-conformer RMSD (MCS atoms): "
        + ", ".join(f"{r:.3f}" for r in per_conf_rmsd)
    )


In [ ]:
view = py3Dmol.view()

for mol in mols:
    view.addModel(Chem.MolToMolBlock(mol), 'sdf')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.2}})
view.show()

In [ ]:
# Add excluded volumes around the list of molecules (all conformations are taken into account)

n_excl = add_excluded_volume(
    mols,
    pharmacophore,
    shell_inner=3.5,     # start 3.5 Å outside the vdW surface
    shell_outer=5.5,     # end   5.5 Å outside the vdW surface
    spacing=2.5,         # grid step (Å)
    feature_clearance=2.5,  # drop grid points within 1.5 Å of an existing feature
    # max_excl=0 by default (no cap); pass max_excl=512 etc. to limit count
)
print(f"\nAdded {n_excl} EXCL spheres around phenol "
      f"({len(pharmacophore)} features total).")


In [ ]:
view = py3Dmol.view()
mol_from_ph = pharmacophore.to_mol()
for mol in mols:
    view.addModel(Chem.MolToMolBlock(mol_from_ph), 'sdf')
    view.setStyle({'stick': {}, 'sphere': {'radius': 0.2}})
view.show()

In [ ]:
# Download a 10,000 compound dataset
MAX_COMPOUNDS = 2000   # None if all

# Define the URL of the file on GitHub
SMI_FILE_URL = "https://raw.githubusercontent.com/silicos-it/PyPharao/master/examples/datasets/compounds_10k.smi"

# Define the local path where the file should be saved
datasets_dir = Path("./datasets")
datasets_dir.mkdir(exist_ok=True) # Create the datasets directory if it doesn't exist
SMI_FILE = datasets_dir / "compounds_10k.smi"

# Download the file
print(f"Downloading {SMI_FILE_URL} to {SMI_FILE}...")
response = requests.get(SMI_FILE_URL)
response.raise_for_status() # Raise an exception for HTTP errors

# Save the file content
with open(SMI_FILE, "wb") as f:
    f.write(response.content)

print(f"Successfully downloaded {SMI_FILE.name} to {SMI_FILE}")

smiles_lines = [
    ln.strip()
    for ln in SMI_FILE.read_text(encoding="utf-8").splitlines()
    if ln.strip()
]
if MAX_COMPOUNDS is not None:
    smiles_lines = smiles_lines[:MAX_COMPOUNDS]

In [ ]:
# Build 3D conformations
NUM_CONFS = 1

print(f"\nBuilding 3D structures for {len(smiles_lines)} SMILES from {SMI_FILE.name}")
prepared: list[tuple[int, str, Chem.Mol]] = []
parse_fail = embed_fail = 0
with tqdm(smiles_lines, desc="3D structures", unit="mol") as pbar:
    for line_idx, smi in enumerate(pbar):
        mol_3d = Chem.MolFromSmiles(smi)
        if mol_3d is None:
            parse_fail += 1
            continue
        mol_3d = Chem.AddHs(mol_3d)
        cids = AllChem.EmbedMultipleConfs(mol_3d, numConfs=NUM_CONFS)
        if len(cids) == 0:
            embed_fail += 1
            continue
        for cid in cids:
            AllChem.MMFFOptimizeMolecule(mol_3d, confId=cid)
        prepared.append((line_idx, smi, mol_3d))
        pbar.set_postfix(ready=len(prepared), parse=parse_fail, embed=embed_fail)

print(
    f"Done: {len(prepared)} ligands ready, "
    f"{parse_fail} invalid SMILES, {embed_fail} embed failures"
)


In [ ]:
# ------------------------------------------------------------
# Run the screen and report the top hits
# ------------------------------------------------------------
#
# `PharmacophoreSearch` enforces excluded volumes on two levels:
#   * `with_exclusion=True`  — soft Pharao volume penalty (subtracted from the
#                              aligned overlap; reflected by `excl_volume`).
#   * `excl_hard_filter=True` (default) — after alignment, every heavy atom is
#                              treated as a sphere of its van der Waals radius;
#                              the hit is rejected if any heavy-atom vdW sphere
#                              swallows a query EXCL marker, i.e. whenever
#                              dist(atom, EXCL) < vdW(atom) + excl_clash_radius.
#                              Pass `excl_hard_filter=False` to recover the pure
#                              soft-penalty behaviour; raise `excl_clash_radius`
#                              to enforce a larger buffer around each EXCL.
#
# The soft penalty is very slow, since all EXCL spheres need to be matched.
# So we will only perform a hard filter here.

hard_searcher = PharmacophoreSearch(pharmacophore)  # excl_hard_filter=True

print("Hard EXCL atom-clash filter ON (default)")
hard_hits = hard_searcher.screen(prepared, progress=True)
hard_sorted = sort_match_results(hard_hits, sort="descending", key="tanimoto")
print_match_results(hard_sorted, limit=10)

SDF_FILE = Path("./top_hits_phenol_excl.sdf")
write_hits_sdf(hard_sorted[:5], SDF_FILE, pharmacophore=pharmacophore)
PDB_FILE = Path("./top_hits_phenol_excl.pdb")
write_hits_pdb(hard_sorted[:5], PDB_FILE, pharmacophore=pharmacophore)


In [ ]:
view = py3Dmol.view()


suppl = Chem.SDMolSupplier(Path("./top_hits_phenol_excl.sdf"))
for mol in suppl:
    if mol is None: continue
    view.addModel(Chem.MolToMolBlock(mol), 'sdf')

view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.2}})
view.show()